<a href="https://colab.research.google.com/github/10dimensions/gnc-toolbox/blob/main/perturbation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np

In [1]:
# --- Physical Constants ---
MU = 398600.4418       # Earth's gravitational parameter (km^3/s^2)
R_EARTH = 6378.137     # Earth's equatorial radius (km)
J2 = 1.08263e-3        # Earth's J2 oblateness coefficient
SECONDS_PER_DAY = 86400.0

In [2]:
def calculate_j2_rates(a_km, e, i_deg):
    """
    Calculates the secular drift rates for RAAN and Argument of Perigee
    due to Earth's J2 oblateness.
    """
    i_rad = np.radians(i_deg)
    p = a_km * (1 - e**2) # Semi-latus rectum
    n = np.sqrt(MU / a_km**3) # Mean motion (rad/s)

    # Secular rate of RAAN (rad/s)
    omega_dot_raan = -1.5 * J2 * (R_EARTH / p)**2 * n * np.cos(i_rad)

    # Secular rate of Argument of Perigee (rad/s)
    omega_dot_argp = 0.75 * J2 * (R_EARTH / p)**2 * n * (4 - 5 * np.sin(i_rad)**2)

    # Convert to degrees per day for readability
    raan_deg_per_day = np.degrees(omega_dot_raan) * SECONDS_PER_DAY
    argp_deg_per_day = np.degrees(omega_dot_argp) * SECONDS_PER_DAY

    return raan_deg_per_day, argp_deg_per_day

In [3]:
def find_sso_inclination(altitude_km, e=0.0):
    """
    Finds the required inclination for a Sun-Synchronous Orbit (SSO).
    An SSO requires the RAAN to precess by ~0.9856 degrees per day
    (360 degrees / 365.2422 days) to match the Earth's orbit around the Sun.
    """
    a_km = R_EARTH + altitude_km
    p = a_km * (1 - e**2)
    n = np.sqrt(MU / a_km**3)

    # Target RAAN drift (rad/s)
    target_raan_rate = np.radians(360.0 / 365.2422) / SECONDS_PER_DAY

    # Solve for cos(i): target = -1.5 * J2 * (R/p)^2 * n * cos(i)
    cos_i = target_raan_rate / (-1.5 * J2 * (R_EARTH / p)**2 * n)

    if abs(cos_i) > 1.0:
        return None # Impossible at this altitude

    i_rad = np.arccos(cos_i)
    return np.degrees(i_rad)

In [4]:
def calculate_initial_drag_decay(alt_km, area_m2, mass_kg):
    """
    Calculates the initial orbital decay rate due to atmospheric drag.
    Uses a simplified exponential atmosphere model for conceptual demonstration.
    """
    # Simplified exponential atmosphere parameters
    rho0 = 1.225 # kg/m^3 (sea level density)
    H = 50.0     # Scale height in km (average for LEO)

    # Density at altitude (kg/m^3)
    rho = rho0 * np.exp(-alt_km / H)

    # Orbital velocity (m/s)
    r_m = (R_EARTH + alt_km) * 1000
    v = np.sqrt(MU * 1e9 / r_m) # Convert MU to m^3/s^2

    # Drag acceleration (m/s^2)
    Cd = 2.2 # Typical drag coefficient for a satellite
    a_drag = 0.5 * rho * v**2 * Cd * (area_m2 / mass_kg)

    # Rate of change of semi-major axis (m/s)
    # da/dt = -2 * a * a_drag / v  (assuming circular orbit where a ≈ r)
    da_dt = -2 * r_m * a_drag / v

    # Convert to km per day
    decay_rate_km_day = (da_dt * SECONDS_PER_DAY) / 1000

    return decay_rate_km_day

In [7]:
# =========================================================================
# Main Execution: Mission Design Analysis
# =========================================================================
if __name__ == "__main__":
    print("--- GNC Perturbation & Orbit Design Utilities ---\n")

    # 1. J2 Rates for a typical LEO satellite (e.g., ISS)
    alt_iss = 420 # km
    a_iss = R_EARTH + alt_iss
    e_iss = 0.0001
    i_iss = 51.6

    raan, argp = calculate_j2_rates(a_iss, e_iss, i_iss)
    print(f"1. ISS J2 Perturbations (Alt: {alt_iss} km, Inc: {i_iss} deg):")
    print(f"   RAAN Drift:       {raan:.4f} deg/day (Massive plane twisting!)")
    print(f"   Arg of Perigee:   {argp:.4f} deg/day\n")

    # 2. Finding the Sun-Synchronous Orbit (SSO) Inclination
    alt_sso = 600 # km
    sso_inc = find_sso_inclination(alt_sso)
    print(f"2. Sun-Synchronous Orbit Design (Alt: {alt_sso} km):")
    if sso_inc:
        print(f"   Required Inclination: {sso_inc:.2f} degrees")
        print(f"   -> Notice it's > 90 deg! SSOs are technically retrograde orbits.\n")

    # 3. The Critical Inclination (Frozen Orbit)
    print("3. The Critical Inclination (Frozen Orbit):")
    print("   Finding the inclination where Arg of Perigee drift is ZERO...")
    # Math: 4 - 5*sin^2(i) = 0  =>  sin(i) = sqrt(4/5)
    i_crit = np.degrees(np.arcsin(np.sqrt(4/5)))
    print(f"   Critical Inclination: {i_crit:.2f} degrees\n")

    # 4. Drag Decay: LEO vs GEO
    print("4. Atmospheric Drag Decay Rates (Area: 10m^2, Mass: 500kg):")
    decay_leo = calculate_initial_drag_decay(400, 10.0, 500)
    decay_geo = calculate_initial_drag_decay(35786, 10.0, 500)
    print(f"   LEO (400 km) decay rate:   {decay_leo:.4f} km/day")
    print(f"   GEO (35786 km) decay rate: {decay_geo:.2e} km/day (Effectively zero!)")

--- GNC Perturbation & Orbit Design Utilities ---

1. ISS J2 Perturbations (Alt: 420 km, Inc: 51.6 deg):
   RAAN Drift:       -4.9510 deg/day (Massive plane twisting!)
   Arg of Perigee:   3.7029 deg/day

2. Sun-Synchronous Orbit Design (Alt: 600 km):
   Required Inclination: 97.79 degrees
   -> Notice it's > 90 deg! SSOs are technically retrograde orbits.

3. The Critical Inclination (Frozen Orbit):
   Finding the inclination where Arg of Perigee drift is ZERO...
   Critical Inclination: 63.43 degrees

4. Atmospheric Drag Decay Rates (Area: 10m^2, Mass: 500kg):
   LEO (400 km) decay rate:   -81202745.2034 km/day
   GEO (35786 km) decay rate: -8.86e-300 km/day (Effectively zero!)
